# Phase 3 — Analytical Track Development and Validation

**Project:** POC-32 — Infocreon Aether Pulse Air Quality Heatmap

**Selected track:** Track A — Comparative Intelligence

**Data version:** `phase3-v1.1`

**Method version:** `track-a-comparative-v1.0.0`

## Purpose

This notebook documents the execution and technical validation of descriptive city-pollutant comparisons using the canonical synthetic dataset.

The results describe the supplied synthetic sample only. They do not establish real-world air quality, regulatory compliance, health risk, causation, or predictive performance.

## 1. Analytical question and method

**Analytical question:** Within each pollutant, how do sampled city measurements compare with each other and with that pollutant's overall mean?

The Track A method groups observations by city and pollutant. It calculates descriptive statistics, compares each group mean with the pollutant-wide mean, and creates deterministic rankings.

The pollutant-wide mean is a descriptive baseline, not a regulatory threshold. All observations are synthetic, so rankings must not be interpreted as real-world air-quality rankings.

In [29]:

from pathlib import Path
import json
import pandas as pd
import subprocess
import sys

# Locate the Phase 3 repository root.
ROOT = Path.cwd().resolve()

for candidate in (ROOT, *ROOT.parents):
    if (
        (candidate / "data-science").is_dir()
        and (candidate / "data").is_dir()
        and (candidate / "docs").is_dir()
    ):
        ROOT = candidate
        break
else:
    raise RuntimeError(
        "Could not locate the repository root. "
        "Open the Phase 3 project folder in VS Code."
    )

DATA = ROOT / "data" / "canonical" / "intelligence_data.csv"
OUT = ROOT / "data-science" / "outputs"

print("Repository root:", ROOT)
print("Canonical dataset exists:", DATA.is_file())
print("Output directory exists:", OUT.is_dir())


Repository root: C:\Users\ACER\Documents\POC-32-AirQualityHeatmap-ParvathyCR1\POC-32-AirQualityHeatMap-ParvathyCR-Phase3
Canonical dataset exists: True
Output directory exists: True


In [30]:

# Load the canonical dataset.
df = pd.read_csv(DATA)

print("Canonical input:", DATA)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Column names:")
print(df.columns.tolist())

display(df.head())


Canonical input: C:\Users\ACER\Documents\POC-32-AirQualityHeatmap-ParvathyCR1\POC-32-AirQualityHeatMap-ParvathyCR-Phase3\data\canonical\intelligence_data.csv
Rows: 96
Columns: 20
Column names:
['record_id', 'record_type', 'observed_at', 'entity_id', 'related_entity_id', 'entity_name', 'category', 'subcategory', 'status', 'stage', 'metric_name', 'metric_value', 'metric_unit', 'text_value', 'latitude', 'longitude', 'source_name', 'source_record_id', 'is_synthetic', 'data_version']


,record_id,record_type,observed_at,entity_id,related_entity_id,entity_name,category,subcategory,status,stage,metric_name,metric_value,metric_unit,text_value,latitude,longitude,source_name,source_record_id,is_synthetic,data_version
0,aq-0001,measurement,2026-08-23T00:00:00Z,ahmedabad,NaN,Ahmedabad,air_quality,no2,observed,observation,no2,44.12,µg/m³,India,23.0225,72.5714,Synthetic OpenAQ-shaped sample,ahmedabad-no2-0000,True,phase3-v1.1
1,aq-0002,measurement,2026-08-23T04:00:00Z,ahmedabad,NaN,Ahmedabad,air_quality,no2,observed,observation,no2,39.33,µg/m³,India,23.0225,72.5714,Synthetic OpenAQ-shaped sample,ahmedabad-no2-0001,True,phase3-v1.1
2,aq-0003,measurement,2026-08-23T20:00:00Z,ahmedabad,NaN,Ahmedabad,air_quality,no2,observed,observation,no2,41.11,µg/m³,India,23.0225,72.5714,Synthetic OpenAQ-shaped sample,ahmedabad-no2-0002,True,phase3-v1.1
3,aq-0004,measurement,2026-08-23T00:00:00Z,ahmedabad,NaN,Ahmedabad,air_quality,o3,observed,observation,o3,72.44,µg/m³,India,23.0225,72.5714,Synthetic OpenAQ-shaped sample,ahmedabad-o3-0003,True,phase3-v1.1
4,aq-0005,measurement,2026-08-23T04:00:00Z,ahmedabad,NaN,Ahmedabad,air_quality,o3,observed,observation,o3,69.92,µg/m³,India,23.0225,72.5714,Synthetic OpenAQ-shaped sample,ahmedabad-o3-0004,True,phase3-v1.1


In [31]:

# Check the distinct city identifiers in the full canonical dataset.
print("Unique entity IDs:", df["entity_id"].nunique())
print("\nEntity IDs:")
print(sorted(df["entity_id"].dropna().unique()))

print("\nCity names and their IDs:")
print(
    df[["entity_id", "entity_name"]]
    .drop_duplicates()
    .sort_values("entity_id")
    .to_string(index=False)
)


Unique entity IDs: 8

Entity IDs:
['ahmedabad', 'bengaluru', 'chennai', 'delhi', 'hyderabad', 'kolkata', 'mumbai', 'pune']

City names and their IDs:
entity_id entity_name
ahmedabad   Ahmedabad
bengaluru   Bengaluru
  chennai     Chennai
    delhi       Delhi
hyderabad   Hyderabad
  kolkata     Kolkata
   mumbai      Mumbai
     pune        Pune


## 3. Canonical input validation

The canonical dataset is the sole analytical input for this notebook. Before running Track A, we verify its expected dimensions, city and pollutant coverage, timestamp coverage, record identifier uniqueness, data version, and synthetic-data designation.

These checks establish structural readiness for descriptive analysis. They do not establish that the synthetic observations represent real-world air quality.

In [32]:

# Validate canonical dataset structure and provenance.

required_columns = [
    "record_id",
    "observed_at",
    "entity_id",
    "entity_name",
    "subcategory",
    "metric_name",
    "metric_value",
    "metric_unit",
    "data_version",
    "is_synthetic",
]

missing_columns = [col for col in required_columns if col not in df.columns]

assert not missing_columns, f"Missing required columns: {missing_columns}"
assert len(df) == 96, f"Expected 96 rows, found {len(df)}"
assert len(df.columns) == 20, f"Expected 20 columns, found {len(df.columns)}"
assert df["entity_id"].nunique() == 8, "Expected 8 distinct cities"
assert df["subcategory"].nunique() == 4, "Expected 4 pollutants"
assert df["record_id"].is_unique, "Duplicate record IDs found"
assert df["data_version"].nunique() == 1, "Multiple data versions found"
assert df["data_version"].iloc[0] == "phase3-v1.1", "Unexpected data version"

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Cities:", df["entity_id"].nunique())
print("Pollutants:", sorted(df["subcategory"].unique()))
print("Data version:", df["data_version"].iloc[0])
print("Record IDs unique:", df["record_id"].is_unique)
print("Structural validation: PASS")


Rows: 96
Columns: 20
Cities: 8
Pollutants: ['no2', 'o3', 'pm10', 'pm2.5']
Data version: phase3-v1.1
Record IDs unique: True
Structural validation: PASS


## 4. Track A — Comparative Intelligence

The analysis compares sampled city measurements within each pollutant. For each city-pollutant group, it calculates the number of observations, mean, median, minimum, and maximum. It then compares the group mean with the pollutant-wide mean and produces deterministic descriptive rankings.

The analysis is implemented in the reusable Track A script. This notebook invokes the project scripts rather than duplicating the analytical implementation.

In [33]:

# Execute the analysis, independent validation, and export verification.
scripts = [
    ROOT / "data-science" / "scripts" / "run_analytical_track.py",
    ROOT / "data-science" / "scripts" / "validate_analytical_track.py",
    ROOT / "data-science" / "scripts" / "export_intelligence_results.py",
]

for script in scripts:
    if not script.is_file():
        raise FileNotFoundError(f"Required script not found: {script}")

    print(f"\nRunning: {script.name}")

    result = subprocess.run(
        [sys.executable, str(script)],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )

    print(result.stdout)

    if result.stderr:
        print("Standard error:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"{script.name} failed with exit code {result.returncode}"
        )

print("\nTrack A execution pipeline: PASS")



Running: run_analytical_track.py
Input rows: 96
Data version: phase3-v1.1
Result groups: 32
Method version: track-a-comparative-v1.0.0
Generated intelligence_results.json and intelligence_summary.json


Running: validate_analytical_track.py
{
  "data_version": "phase3-v1.1",
  "method_version": "track-a-comparative-v1.0.0",
  "input_rows": 96,
  "input_columns": 20,
  "expected_group_count": 32,
  "actual_result_count": 32,
  "maximum_absolute_mean_error": 0.0,
  "maximum_absolute_baseline_gap_error": 0.0,
  "group_size_errors": [],
  "ranking_errors": [],
  "checks": {
    "canonical_rows_96": true,
    "canonical_columns_20": true,
    "expected_data_version": true,
    "all_records_synthetic": true,
    "unique_record_ids": true,
    "expected_group_count": true,
    "independent_means_match": true,
    "independent_baseline_gaps_match": true,
    "group_sizes_match": true,
    "result_version_consistent": true,
    "method_version_present": true,
    "summary_result_count_matches"

## 5. Results and technical validation

The Track A pipeline generates 32 city-pollutant comparison groups from the 96-row canonical dataset.

Validation independently checks the descriptive means and pollutant-baseline gaps, group sizes, deterministic ranking order, data-version consistency, method-version metadata, and summary result count.

A passing technical validation supports the correctness of the implemented calculations against the canonical input. It does not establish real-world representativeness or predictive validity.

In [34]:

# Load the saved analytical outputs.
RESULTS_PATH = OUT / "intelligence_results.json"
SUMMARY_PATH = OUT / "intelligence_summary.json"
VALIDATION_PATH = OUT / "validation_metrics.json"
WEAK_CASE_PATH = OUT / "weak_case_review.json"

results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
validation = json.loads(VALIDATION_PATH.read_text(encoding="utf-8"))
weak_cases = json.loads(WEAK_CASE_PATH.read_text(encoding="utf-8"))

print("Result records:", len(results))
print("Summary result count:", summary["result_count"])
print("Data version:", summary["data_version"])
print("Method version:", summary["method_version"])
print("Validation result:", validation["validation_result"])
print("Failed checks:", validation["failed_checks"])
print("Issues:", validation["issues"])

assert len(results) == 32
assert summary["result_count"] == 32
assert summary["data_version"] == "phase3-v1.1"
assert summary["method_version"] == "track-a-comparative-v1.0.0"
assert validation["validation_result"] == "PASS"
assert not validation["failed_checks"]
assert not validation["issues"]

print("\nSaved output evidence: PASS")


Result records: 32
Summary result count: 32
Data version: phase3-v1.1
Method version: track-a-comparative-v1.0.0
Validation result: PASS
Failed checks: []
Issues: []

Saved output evidence: PASS


In [35]:

# Display the descriptive findings by pollutant.
findings = pd.DataFrame(summary["key_findings"])

display(findings)

print(
    "\nInterpretation: these comparisons describe the synthetic sample only. "
    "They are not real-world air-quality rankings or regulatory assessments."
)


,pollutant,highest_sampled_city_mean,highest_mean_value,lowest_sampled_city_mean,lowest_mean_value,pollutant_baseline_mean,finding_scope
0,no2,Delhi,52.893333,Bengaluru,23.116667,36.207500,Synthetic descriptive comparison only
1,o3,Delhi,90.466667,Bengaluru,42.026667,62.297083,Synthetic descriptive comparison only
2,pm10,Delhi,118.643333,Bengaluru,56.206667,81.862917,Synthetic descriptive comparison only
3,pm2.5,Delhi,65.416667,Bengaluru,29.420000,44.904167,Synthetic descriptive comparison only



Interpretation: these comparisons describe the synthetic sample only. They are not real-world air-quality rankings or regulatory assessments.
